# Provenance-aware Representation Learning on a Multi-source Urban Graph
### Companion notebook for the Representation Learning Benchmark section

This extends the first notebook (which covered the shallow-embedding tutorial and the node-classification
method spectrum) to the pieces that are specific to the **multi-source, provenance-aware** benchmark:

- a **multi-source synthetic graph** with *canonical* entities and *source-specific evidence*
  (CityGML / OSM / ML-derived), linked by **confidence-weighted** edges, plus **coverage** features;
- the **provenance-agnostic vs. provenance-aware** contrast, controlled so the only difference is the
  provenance information the encoder may use;
- **Task T2** node classification (roof material) and **Task T3** cross-source matching (link prediction /
  entity resolution) with hard negatives;
- a short **cross-city transfer** demo (train on city A, test on city B).

It runs offline on synthetic data; swap in your real graph the same way as in notebook 1. Exports the
figures named by the paper section: `fig_prov_results.pdf`, `fig_matching_pr.pdf`, `fig_crosscity.pdf`,
`fig_prov_tsne.pdf`.


In [2]:
# %pip install torch torch_geometric scikit-learn matplotlib pandas numpy   # (see notebook 1)
import time, numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             average_precision_score, precision_recall_curve)
from torch_geometric.nn import GraphConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
print("device:", device)

device: cuda


/home/luarzou/Projects/aussenraume/KDD_benchmark_paper/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. The provenance data model

Each real building is a **canonical entity**. Every source that observes it contributes an **evidence
node** (authoritative CityGML, crowd-sourced OSM, ML-derived), attached to the canonical entity by an edge
whose weight is the **correspondence confidence** (high for authoritative, noisier for crowd/ML). Not every
source covers every building, so **coverage** varies. Derived evidence never replaces authoritative
values; it is additional evidence.

The two regimes we compare:

- **Provenance-agnostic:** flatten to canonical entities, drop source identity and confidence.
- **Provenance-aware:** keep source-type features, use confidence as edge weights (Eq. 1 in the paper),
  and expose coverage. We keep the *architecture identical* and only toggle this information, so any gap
  is due to provenance awareness, not capacity.


In [ ]:
# ---------------- config ----------------
N_BUILDINGS = 600
GRID        = 20
P_CG, P_OSM, P_ML = 0.95, 0.80, 0.55          # source coverage probabilities
HID, LR, EPOCHS, SEEDS = 64, 0.01, 150, [0, 1, 2]
K_ADJ = 6

In [ ]:
def make_multisource_city(n=N_BUILDINGS, grid=GRID, seed=0, region_shift=0.0):
    from sklearn.neighbors import NearestNeighbors
    rng = np.random.default_rng(seed)

    coords = rng.uniform(0, grid, (n, 2))
    height = rng.gamma(2, 4, n) + 3.0
    area   = rng.gamma(2, 60, n) + 40.0
    orient = rng.uniform(0, 180, n)

    # roof material: spatially autocorrelated (region_shift rotates the layout -> "another city")
    qx = (((coords[:, 0] + region_shift) % grid) > grid / 2).astype(int)
    qy = (coords[:, 1] > grid / 2).astype(int)
    region = qx * 2 + qy
    material = (np.eye(4)[region] * 2.5 + rng.normal(0, 1.0, (n, 4))).argmax(1)

    geo = np.stack([height, area, orient], 1)                     # authoritative geometry
    has_cg  = rng.random(n) < P_CG
    has_osm = rng.random(n) < P_OSM
    has_ml  = rng.random(n) < P_ML
    coverage = has_cg.astype(float) + has_osm + has_ml

    SRC = {"canon": 0, "cityGML": 1, "OSM": 2, "ML": 3}
    rows_feat, rows_src, cg_id, osm_id, ml_id = [], [], -np.ones(n, int), -np.ones(n, int), -np.ones(n, int)

    def add_node(g3, cov, src):
        rows_feat.append(list(g3) + [cov])
        rows_src.append(src)
        return len(rows_feat) - 1

    # canonical nodes = indices 0..n-1 (geometry = noisy summary; coverage feature set)
    for i in range(n):
        add_node(geo[i] + rng.normal(0, 0.5, 3), coverage[i], SRC["canon"])
    # evidence nodes
    for i in range(n):
        if has_cg[i]:  cg_id[i]  = add_node(geo[i] + rng.normal(0, 0.3, 3), 0.0, SRC["cityGML"])
        if has_osm[i]: osm_id[i] = add_node(geo[i] + rng.normal(0, 2.0, 3) + 1.0, 0.0, SRC["OSM"])
        if has_ml[i]:  ml_id[i]  = add_node(geo[i] + rng.normal(0, 1.0, 3), 0.0, SRC["ML"])

    feat = np.array(rows_feat, np.float32)
    feat[:, :3] = (feat[:, :3] - feat[:, :3].mean(0)) / (feat[:, :3].std(0) + 1e-6)   # z-score geometry
    src = np.array(rows_src, int)
    src_onehot = np.eye(4, dtype=np.float32)[src]
    x = np.concatenate([feat, src_onehot], 1)                     # [geo(3), coverage(1), src(4)] = 8 dims
    SRC_COLS = list(range(4, 8))                                  # source-type columns (zeroed when agnostic)

    # ---- edges (bidirectional) with confidence weights ----
    ei, ew = [], []
    def link(a, b, w):
        ei.append((a, b)); ew.append(w); ei.append((b, a)); ew.append(w)
    for i in range(n):
        if cg_id[i]  >= 0: link(i, cg_id[i],  rng.uniform(0.85, 0.98))   # authoritative: high confidence
        if osm_id[i] >= 0: link(i, osm_id[i], rng.uniform(0.40, 0.80))   # crowd: noisier
        if ml_id[i]  >= 0: link(i, ml_id[i],  rng.uniform(0.50, 0.90))
    nn = NearestNeighbors(n_neighbors=K_ADJ + 1).fit(coords)
    _, idx = nn.kneighbors(coords)
    for i in range(n):
        for j in idx[i, 1:]:
            link(i, int(j), 1.0)                                  # spatial adjacency (structural)

    G = dict(
        x=torch.tensor(x), src_cols=SRC_COLS,
        edge_index=torch.tensor(np.array(ei).T, dtype=torch.long),
        edge_weight=torch.tensor(np.array(ew), dtype=torch.float),
        y=torch.tensor(material, dtype=torch.long), n_canon=n,
        pos=torch.tensor(coords, dtype=torch.float),
        cg_id=cg_id, osm_id=osm_id, adj=idx,
    )
    return G

G = make_multisource_city(seed=0)
print("nodes:", G["x"].shape[0], "| features:", G["x"].shape[1],
      "| edges:", G["edge_index"].shape[1], "| canonical:", G["n_canon"])

## 2. Encoder and the agnostic/aware switch

A 2-layer `GraphConv` encoder (mean aggregation) that optionally uses edge confidence. `make_inputs`
implements the switch: **aware** keeps the source-type columns and passes confidence as `edge_weight`;
**agnostic** zeroes the source columns and drops the weights. Architecture and parameter count are
identical.


In [ ]:
class Encoder(torch.nn.Module):
    def __init__(self, d_in, hid):
        super().__init__()
        self.c1 = GraphConv(d_in, hid, aggr="mean")
        self.c2 = GraphConv(hid, hid, aggr="mean")
    def forward(self, x, ei, ew=None):
        x = self.c1(x, ei, ew).relu()
        x = self.c2(x, ei, ew).relu()
        return x

def make_inputs(G, aware):
    x = G["x"].clone()
    if not aware:
        x[:, G["src_cols"]] = 0.0                 # hide source identity
    ew = G["edge_weight"] if aware else None       # hide confidence
    return x.to(device), G["edge_index"].to(device), (ew.to(device) if ew is not None else None)

## 3. Task T2 — node classification (roof material), agnostic vs. aware
Canonical nodes are indices `0..n_canon-1`. We train end-to-end and report macro-F1 over seeds.


In [ ]:
def split_canon(n, seed, frac=(0.6, 0.2, 0.2)):
    rng = np.random.default_rng(seed); idx = rng.permutation(n)
    a, b = int(frac[0]*n), int((frac[0]+frac[1])*n)
    f = lambda ix: torch.tensor(np.isin(np.arange(n), ix))
    return f(idx[:a]), f(idx[a:b]), f(idx[b:])

def run_nodeclf(G, aware, seed):
    set_seed(seed)
    x, ei, ew = make_inputs(G, aware)
    y = G["y"].to(device); n = G["n_canon"]
    tr, va, te = (m.to(device) for m in split_canon(n, seed))
    enc = Encoder(x.size(1), HID).to(device)
    head = torch.nn.Linear(HID, int(G["y"].max())+1).to(device)
    opt = torch.optim.Adam(list(enc.parameters())+list(head.parameters()), lr=LR, weight_decay=5e-4)
    for _ in range(EPOCHS):
        enc.train(); opt.zero_grad()
        logit = head(enc(x, ei, ew)[:n])
        F.cross_entropy(logit[tr], y[tr]).backward(); opt.step()
    enc.eval()
    with torch.no_grad():
        pred = head(enc(x, ei, ew)[:n]).argmax(1)
    yt, yp = y[te].cpu().numpy(), pred[te].cpu().numpy()
    return {"acc": accuracy_score(yt, yp), "macro_f1": f1_score(yt, yp, average="macro")}

RESULTS = []
print("T2 roof-material classification:")
for aware in [False, True]:
    per = [run_nodeclf(G, aware, s) for s in SEEDS]
    mf1 = np.mean([p["macro_f1"] for p in per]); sd = np.std([p["macro_f1"] for p in per])
    RESULTS.append({"task": "T2 node-clf", "regime": "aware" if aware else "agnostic",
                    "metric": "macro_f1", "value": mf1, "std": sd})
    print(f"  {'aware' if aware else 'agnostic':9s}  macroF1 = {mf1:.3f} ± {sd:.3f}")

## 4. Task T3 — cross-source matching (link prediction)

**Positives**: (CityGML evidence of building $i$, OSM evidence of building $i$). **Hard negatives**:
(CityGML of $i$, OSM of a *spatial neighbour* $j\neq i$) — the confusable case. A shared encoder embeds all
nodes; the decoder scores a pair by dot product. We report ROC-AUC, average precision, and Hits@1
(rank each CityGML node's true OSM match against its hard negatives).


In [ ]:
def build_pairs(G, seed=0, n_neg_per_pos=1):
    rng = np.random.default_rng(seed)
    cg, osm, adj = G["cg_id"], G["osm_id"], G["adj"]
    pos, neg, rank_sets = [], [], []
    for i in range(G["n_canon"]):
        if cg[i] < 0 or osm[i] < 0:
            continue
        pos.append((cg[i], osm[i]))
        negs_i = []
        for j in adj[i, 1:]:
            if osm[j] >= 0 and j != i:
                negs_i.append(osm[j])
        if not negs_i:
            continue
        for j in rng.choice(negs_i, size=min(n_neg_per_pos, len(negs_i)), replace=False):
            neg.append((cg[i], int(j)))
        rank_sets.append((cg[i], osm[i], negs_i[:8]))   # for Hits@1
    return np.array(pos), np.array(neg), rank_sets

def run_matching(G, aware, seed):
    set_seed(seed)
    x, ei, ew = make_inputs(G, aware)
    pos, neg, rank_sets = build_pairs(G, seed=seed)
    pairs = np.vstack([pos, neg]); labels = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
    rng = np.random.default_rng(seed); perm = rng.permutation(len(pairs))
    pairs, labels = pairs[perm], labels[perm]
    a, b = int(0.6*len(pairs)), int(0.8*len(pairs))
    tr, va, te = slice(0, a), slice(a, b), slice(b, None)

    enc = Encoder(x.size(1), HID).to(device)
    opt = torch.optim.Adam(enc.parameters(), lr=LR, weight_decay=5e-4)
    P = torch.tensor(pairs, dtype=torch.long, device=device)
    L = torch.tensor(labels, dtype=torch.float, device=device)
    def scores(z, idx):
        za, zb = z[P[idx, 0]], z[P[idx, 1]]
        return (za * zb).sum(-1)
    for _ in range(EPOCHS):
        enc.train(); opt.zero_grad()
        z = enc(x, ei, ew)
        F.binary_cross_entropy_with_logits(scores(z, tr), L[tr]).backward(); opt.step()
    enc.eval()
    with torch.no_grad():
        z = enc(x, ei, ew)
        s_te = torch.sigmoid(scores(z, te)).cpu().numpy(); y_te = labels[te]
        auc = roc_auc_score(y_te, s_te); ap = average_precision_score(y_te, s_te)
        # Hits@1
        hit = 0; tot = 0
        for cg_i, osm_true, negs in rank_sets:
            cand = [osm_true] + list(negs)
            zc = z[cg_i]; sc = (z[torch.tensor(cand, device=device)] * zc).sum(-1)
            if sc.argmax().item() == 0: hit += 1
            tot += 1
        hits1 = hit / max(tot, 1)
        pr, rc, _ = precision_recall_curve(y_te, s_te)
    return {"auc": auc, "ap": ap, "hits1": hits1, "pr": pr, "rc": rc}

print("T3 cross-source matching:")
PR = {}
for aware in [False, True]:
    per = [run_matching(G, aware, s) for s in SEEDS]
    reg = "aware" if aware else "agnostic"
    auc = np.mean([p["auc"] for p in per]); ap = np.mean([p["ap"] for p in per])
    h1  = np.mean([p["hits1"] for p in per])
    PR[reg] = per[0]  # keep one PR curve for the figure
    for metric, val in [("auc", auc), ("ap", ap), ("hits1", h1)]:
        RESULTS.append({"task": "T3 matching", "regime": reg, "metric": metric, "value": val, "std": 0.0})
    print(f"  {reg:9s}  AUC={auc:.3f}  AP={ap:.3f}  Hits@1={h1:.3f}")

## 5. Cross-city transfer (train on city A, test on city B)

We generate a second synthetic city with a shifted material layout and evaluate the encoder trained on A,
without retraining, on B. This is the two-city stand-in for the paper's five-city, three-continent
transfer matrix.


In [ ]:
G_B = make_multisource_city(seed=99, region_shift=GRID/2)   # different layout -> "another city"

def transfer(G_train, G_test, aware, seed):
    set_seed(seed)
    xtr, eitr, ewtr = make_inputs(G_train, aware); ytr = G_train["y"].to(device); ntr = G_train["n_canon"]
    enc = Encoder(xtr.size(1), HID).to(device)
    head = torch.nn.Linear(HID, int(G_train["y"].max())+1).to(device)
    opt = torch.optim.Adam(list(enc.parameters())+list(head.parameters()), lr=LR, weight_decay=5e-4)
    for _ in range(EPOCHS):
        enc.train(); opt.zero_grad()
        F.cross_entropy(head(enc(xtr, eitr, ewtr)[:ntr]), ytr).backward(); opt.step()
    xte, eite, ewte = make_inputs(G_test, aware); yte = G_test["y"].to(device); nte = G_test["n_canon"]
    enc.eval()
    with torch.no_grad():
        pred = head(enc(xte, eite, ewte)[:nte]).argmax(1)
    return f1_score(yte.cpu().numpy(), pred.cpu().numpy(), average="macro")

print("Cross-city transfer (A -> B), roof material macro-F1:")
XCITY = {}
for aware in [False, True]:
    vals = [transfer(G, G_B, aware, s) for s in SEEDS]
    XCITY["aware" if aware else "agnostic"] = (np.mean(vals), np.std(vals))
    print(f"  {'aware' if aware else 'agnostic':9s}  {np.mean(vals):.3f} ± {np.std(vals):.3f}")

## 6. Figures for the paper
Exports `fig_prov_results.pdf`, `fig_matching_pr.pdf`, `fig_crosscity.pdf`, `fig_prov_tsne.pdf`.


In [ ]:
df = pd.DataFrame(RESULTS); df.to_csv("results_provenance.csv", index=False)
print(df.to_string(index=False))

# --- Figure: agnostic vs aware across tasks (T2 macro-F1, T3 AUC) ---
t2 = df[(df.task=="T2 node-clf") & (df.metric=="macro_f1")].set_index("regime")["value"]
t3 = df[(df.task=="T3 matching") & (df.metric=="auc")].set_index("regime")["value"]
labels = ["T2 roof-material\n(macro-F1)", "T3 matching\n(AUC)"]
agn = [t2["agnostic"], t3["agnostic"]]; awa = [t2["aware"], t3["aware"]]
xx = np.arange(2); w = 0.35
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.bar(xx-w/2, agn, w, label="provenance-agnostic")
ax.bar(xx+w/2, awa, w, label="provenance-aware")
ax.set_xticks(xx); ax.set_xticklabels(labels); ax.set_ylim(0, 1); ax.set_ylabel("score")
ax.set_title("Provenance-agnostic vs. provenance-aware"); ax.legend()
fig.tight_layout(); fig.savefig("fig_prov_results.pdf"); plt.show()

In [ ]:
# --- Figure: matching precision-recall, agnostic vs aware ---
fig, ax = plt.subplots(figsize=(5.5, 4.2))
for reg, style in [("agnostic", "--"), ("aware", "-")]:
    ax.plot(PR[reg]["rc"], PR[reg]["pr"], style, label=reg)
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_title("Cross-source matching (hard negatives)"); ax.legend()
fig.tight_layout(); fig.savefig("fig_matching_pr.pdf"); plt.show()

# --- Figure: cross-city transfer bar ---
fig, ax = plt.subplots(figsize=(4.6, 4))
regs = ["agnostic", "aware"]
ax.bar(regs, [XCITY[r][0] for r in regs], yerr=[XCITY[r][1] for r in regs], capsize=4,
       color=["#8888bb", "#33aa66"])
ax.set_ylim(0, 1); ax.set_ylabel("macro-F1 on held-out city (B)")
ax.set_title("Cross-city transfer (A -> B)")
fig.tight_layout(); fig.savefig("fig_crosscity.pdf"); plt.show()

In [ ]:
# --- Figure: t-SNE of canonical embeddings (aware encoder) ---
from sklearn.manifold import TSNE
set_seed(0)
x, ei, ew = make_inputs(G, aware=True)
enc = Encoder(x.size(1), HID).to(device)
head = torch.nn.Linear(HID, int(G["y"].max())+1).to(device)
opt = torch.optim.Adam(list(enc.parameters())+list(head.parameters()), lr=LR, weight_decay=5e-4)
y = G["y"].to(device); n = G["n_canon"]; tr = split_canon(n, 0)[0].to(device)
for _ in range(EPOCHS):
    enc.train(); opt.zero_grad()
    F.cross_entropy(head(enc(x, ei, ew)[:n])[tr], y[tr]).backward(); opt.step()
enc.eval()
with torch.no_grad(): emb = enc(x, ei, ew)[:n].cpu().numpy()

z2 = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(emb)
cov = (G["x"][:n, 3].numpy())
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
s0 = axes[0].scatter(z2[:,0], z2[:,1], c=cov, cmap="viridis", s=10)
axes[0].set_title("coloured by source coverage"); fig.colorbar(s0, ax=axes[0], shrink=0.8)
s1 = axes[1].scatter(z2[:,0], z2[:,1], c=G["y"][:n].numpy(), cmap="tab10", s=10)
axes[1].set_title("coloured by roof material")
for a in axes: a.set_xticks([]); a.set_yticks([])
axes[1].legend(*s1.legend_elements(), title="material", fontsize=8, loc="best")
fig.suptitle("Provenance-aware canonical-entity embeddings")
fig.tight_layout(); fig.savefig("fig_prov_tsne.pdf"); plt.show()

## 7. Moving to the real \benchname{} graph

1. Load canonical entities and per-source evidence nodes from Neo4j into the same arrays: a feature matrix
   `[geometry..., coverage, source-onehot]`, an `edge_index`, and an `edge_weight` holding the real
   correspondence **confidences**; keep `cg_id` / `osm_id` maps for the matching pairs.
2. Real hard negatives already exist in the data (spatially close, non-matching candidates); use them
   instead of the synthetic neighbour sampler.
3. For the full paper: replace the confidence-weighted `GraphConv` with **source-specific** message
   passing (R-GCN / HGT) to realise per-source transforms $\mathbf W_{s(e)}$ in Eq. 1, and expand
   cross-city transfer to the five-city matrix.
4. Add Task T1 (regression: height / storeys; imputation of missing authoritative attributes) by swapping
   the head to a scalar and the loss to `F.mse_loss`.
